# Part 1 — What It Means to Generate Data

_Rigorous Courses · Diffusion Models — Part 1 of 12_

**Every image is a point, every dataset is a cloud, and generating means drawing a brand-new point from the cloud's rule**

In this notebook you hold real image data in your hands: print the actual numbers inside an image, flatten a grid into a vector and back without losing anything, watch a whole dataset become a 2-D cloud with crowded and empty regions, and close with a teaser — one digit drowning in five rising doses of noise.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)

## A picture is a list of numbers

The lesson claims an image is nothing but a grid of brightness numbers — so let's open one up and look at the numbers. We use scikit-learn's small **digits** dataset: 1,797 handwritten digits, each 8 pixels by 8 pixels.

### Step 1 — Load the dataset and check the raw material

The next cell loads all 1,797 images and rescales the pixel values into the 0-to-1 convention from the lesson (0 = black, 1 = white). We assert the range really is 0-to-1 and that the shapes are what we claimed: 1,797 images, each an 8-by-8 grid.

In [ ]:
digits = load_digits()
images = digits.images / 16.0   # rescale brightness into the 0-1 range
labels = digits.target

n_images = images.shape[0]

print(f"number of images: {n_images}")
print(f"shape of one image: {images[0].shape}")
print(f"brightness range: {images.min():.2f} to {images.max():.2f}")

assert images.min() >= 0.0
assert images.max() <= 1.0
assert images.shape == (1797, 8, 8)

### Step 2 — Look at ten digits

Before any math, look at the data as pictures. They are blocky (only 64 pixels each), but every one is clearly a digit — these are the "realistic images" that crowd one thin sliver of a 64-dimensional space.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for ax, image, label in zip(axes, images[:10], labels[:10]):
    ax.imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"label {label}")
    ax.axis("off")
fig.suptitle("ten images from the digits dataset (8x8 pixels each)")
plt.tight_layout()
plt.show()

### Step 3 — Print the numbers inside one image

Now the lesson's central claim, verified by staring at it: the first image (a zero) printed as its raw 8-by-8 grid of brightness numbers. Squint at the printout — the ring of large values with a hole of near-zeros in the middle IS the zero you saw above. There is nothing else in the image.

In [ ]:
one_digit = images[0]

print("the 8x8 grid of brightness numbers for the first image (a zero):")
print(np.round(one_digit, 2))

### Step 4 — Flatten the grid into a vector, and go back

Reading the grid row by row (book order) turns it into a flat list of $8 \times 8 = 64$ numbers — a **vector** in 64-dimensional space. The reshape back must restore the grid exactly: flattening is a change of shape, not of content. The asserts check both directions.

In [ ]:
flat = one_digit.reshape(64)
restored = flat.reshape(8, 8)

print(f"shape before flattening: {one_digit.shape}")
print(f"shape after flattening: {flat.shape}")
print(f"first 8 entries of the vector: {np.round(flat[:8], 2)}")

assert one_digit.shape == (8, 8)
assert flat.shape == (64,)
assert np.array_equal(restored, one_digit)

## Datasets are clouds of points

One image = one point in 64-dimensional space, so 1,797 images = a cloud of 1,797 points. We cannot draw 64 axes, but we can do what the lesson's toy did: pick just 2 of the 64 pixel positions and plot every image by its brightness at those two spots — a 2-D slice through the real cloud.

### Step 1 — Scatter all 1,797 digits in a 2-pixel slice

We pick pixel 20 (row 2, column 4) and pixel 36 (row 4, column 4) — both near the center of the digit, where brightness varies a lot. Pixel values in this dataset come in 17 levels, so many points would stack exactly on top of each other; a tiny random jitter (purely cosmetic) spreads the stacks so you can see how crowded each spot is. Look for structure: heavy bands and corners where points pile up, and regions with almost nothing.

In [ ]:
flat_all = images.reshape(n_images, 64)

pixel_a = 20   # row 2, column 4 of the 8x8 grid
pixel_b = 36   # row 4, column 4

jitter_a = rng.normal(0.0, 0.012, size=n_images)
jitter_b = rng.normal(0.0, 0.012, size=n_images)

plt.figure(figsize=(6, 5))
plt.scatter(flat_all[:, pixel_a] + jitter_a, flat_all[:, pixel_b] + jitter_b,
            s=8, alpha=0.25, color="#4ea1ff")
plt.xlabel("brightness of pixel 20 (row 2, column 4)")
plt.ylabel("brightness of pixel 36 (row 4, column 4)")
plt.title("all 1797 digits, seen through 2 of their 64 pixels")
plt.show()

assert flat_all.shape == (1797, 64)

## The cloud has a shape

The lesson defined a **probability distribution** informally as a rule saying how likely each region is: crowded regions get a big share, empty regions almost none. Let's verify that crowded-vs-empty is a real, measurable feature of this dataset — not a metaphor.

### Step 1 — Histogram one pixel across the whole dataset

The histogram counts how many of the 1,797 images take each brightness value at pixel 20. Some values occur hundreds of times, others almost never — the one-dimensional shadow of "the cloud has crowded and empty regions".

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(flat_all[:, pixel_a], bins=17, color="#4ea1ff", edgecolor="black")
plt.xlabel("brightness of pixel 20")
plt.ylabel("number of images")
plt.title("how often each brightness occurs at pixel 20 (1797 images)")
plt.show()

### Step 2 — Build a miniature $p(x)$: counts per cell

Exactly as in the lesson (and practice problem 5): chop the 2-pixel map into a 4-by-4 grid of cells and count the points in each. Dividing by the total turns counts into **shares** — and legitimate shares must be nonnegative and sum to exactly 1. The asserts check both laws.

In [ ]:
bin_edges = np.array([0.0, 0.25, 0.5, 0.75, 1.0001])
counts, _, _ = np.histogram2d(flat_all[:, pixel_a], flat_all[:, pixel_b],
                              bins=[bin_edges, bin_edges])
counts = counts.astype(int)
total = counts.sum()
shares = counts / total

print("points per cell (rows: pixel 20 dark to bright, columns: pixel 36 dark to bright):")
print(counts)
print(f"total points counted: {total}")
print(f"largest single-cell share: {shares.max():.3f}")
print(f"sum of all 16 shares: {shares.sum():.6f}")

assert total == 1797
assert shares.min() >= 0.0
assert abs(shares.sum() - 1.0) < 1e-12

## A teaser: drowning a digit in noise

The diffusion story starts by destroying structure gradually. Here is the crudest possible version, just to see destruction happen:

$$x_{\text{noisy}} = x_0 + \sigma\,\epsilon$$

Read it out loud: the noisy image is the clean image $x_0$ plus $\sigma$ times a grid of fresh random numbers $\epsilon$ (one per pixel); the knob $\sigma$ sets how big the random nudges are. By part 6 you will replace this crude rule with the precisely scheduled version $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, where the destruction is controlled step by step — for now we only want to watch structure dissolve.

### Step 1 — Make five noisier and noisier versions

Five values of $\sigma$, from 0 (untouched) to 1.6 (drowned). Watch the digit survive small doses and vanish under large ones — display values are clipped to the 0-1 range so the images stay comparable.

In [ ]:
sigmas = [0.0, 0.2, 0.4, 0.8, 1.6]

noisy_versions = []
for sigma in sigmas:
    eps = rng.standard_normal(size=(8, 8))
    noisy = one_digit + sigma * eps
    noisy_versions.append(noisy)

fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
for ax, sigma, noisy in zip(axes, sigmas, noisy_versions):
    ax.imshow(np.clip(noisy, 0.0, 1.0), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"sigma = {sigma}")
    ax.axis("off")
fig.suptitle("one digit drowning in noise (display clipped to the 0-1 range)")
plt.tight_layout()
plt.show()

### Step 2 — Verify the destruction is gradual and monotone

Claim to check: the corruption really grows with $\sigma$. We measure the spread (standard deviation — the typical size) of the difference between each noisy version and the clean image. At $\sigma = 0$ the spread must be exactly 0, each measured spread should sit near its $\sigma$, and the sequence must strictly increase.

In [ ]:
spreads = []
for noisy in noisy_versions:
    corruption = noisy - one_digit
    spreads.append(float(np.std(corruption)))

for sigma, spread in zip(sigmas, spreads):
    print(f"sigma = {sigma:4.1f}   measured spread of (noisy - clean) = {spread:.3f}")

assert spreads[0] == 0.0
assert np.all(np.diff(spreads) > 0)
assert abs(spreads[-1] - 1.6) < 0.4

## Practice

The same problems as the lesson. Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Our 2-pixel toy dataset is A=(0.90, 0.10), B=(0.85, 0.15), C=(0.80, 0.10), D=(0.10, 0.90), E=(0.15, 0.85), F=(0.20, 0.90), each written as (left brightness, right brightness). Three new candidates arrive: G=(0.83, 0.12), H=(0.50, 0.50), K=(0.05, 0.95). For each candidate, compute the distance to its nearest dataset point and decide whether it is a plausible new sample.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
data_points = np.array([
    [0.90, 0.10],
    [0.85, 0.15],
    [0.80, 0.10],
    [0.10, 0.90],
    [0.15, 0.85],
    [0.20, 0.90],
])
candidates = {"G": (0.83, 0.12), "H": (0.50, 0.50), "K": (0.05, 0.95)}

for name, cand in candidates.items():
    diffs = data_points - np.array(cand)
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    print(f"{name}: nearest dataset point is {dists.min():.3f} away")
```

- G: nearest point about **0.036** away (inside the A-B-C cluster; even closer than typical within-cluster spacing of about 0.07) — plausible.
- H: nearest point about **0.495** away (the empty middle; roughly seven times the within-cluster spacing) — not plausible.
- K: nearest point about **0.071** away (right at the edge of the D-E-F cluster) — plausible.

**Answer:** G and K are plausible new samples; H is not — it lies where this dataset's rule puts almost no weight.

</details>

**Problem 2.** A tiny screen shows images of 4 pixels, each pixel taking one of 8 brightness levels. (a) How many distinct images can the screen show? (b) A model that memorized its 100 training images can "generate" at most how many distinct images — and what fraction of all possible images is that?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

Count choice by choice (independent choices multiply): $8 \times 8 = 64$ for two pixels, $64 \times 8 = 512$ for three, $512 \times 8 = 4096$ for all four — that is $8^4$.

```python
n_possible = 8 ** 4
fraction = 100 / n_possible

print(f"distinct 4-pixel, 8-level images: {n_possible}")
print(f"fraction reachable by a 100-image memorizer: {fraction:.4f}")
```

A memorizer replays only what it stored: at most 100 images, which is $100/4096 \approx 0.024$ — about 2.4 percent, with over 97 percent of even this toy space permanently out of reach.

**Answer:** (a) $8^4 = 4096$. (b) At most 100, about 2.4 percent — and for real images the ratio is roughly $10^4$ out of $10^{1888}$: memorization does not scale into generation.

</details>

**Problem 3.** Generative or discriminative? Apply the one-line test — "is the output a judgment about given data, or new data?" — to each task: (a) deciding whether a photo shows a cat or a dog; (b) producing a brand-new face photo; (c) flagging an email as spam; (d) filling in the missing right half of a torn photo; (e) naming the genre of a song clip; (f) turning the sentence "a fox in the snow" into a picture.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) Photo in, one of two labels out — **discriminative**.
- (b) An entire new image out — **generative**.
- (c) Email in, spam-or-not out — **discriminative**.
- (d) The output is new image content that must look like it came from the photo cloud — **generative** (conditional: steered by the visible half).
- (e) Clip in, genre label out — **discriminative**.
- (f) Sentence in, image out — **generative** (conditional: the sentence steers which region of the cloud to sample from; part 12's topic).

**Answer:** (a), (c), (e) discriminative; (b), (d), (f) generative, with (d) and (f) being conditional generation.

</details>

**Problem 4.** A 3-row by 3-column image is flattened row by row (book order) into positions 0 through 8; rows and columns are also counted from 0. (a) Which vector position holds the pixel at row 1, column 2? (b) Which (row, column) does position 7 come from? Work it by hand with the rule position $= 3r + c$, then verify with code.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

By hand: (a) $3 \times 1 + 2 = 5$. (b) $7 \div 3 = 2$ remainder $1$, so row 2, column 1 — and the round trip $3 \times 2 + 1 = 7$ checks out.

```python
grid = np.arange(9).reshape(3, 3)

print("positions after row-by-row flattening:")
print(grid)

pos = 3 * 1 + 2
row = 7 // 3
col = 7 % 3

print(f"(row 1, col 2) lands at position {pos}")
print(f"position 7 comes from (row {row}, col {col})")
```

**Answer:** (a) position 5; (b) row 2, column 1. In code: `pos = 3*row + col`, `row = pos // 3`, `col = pos % 3`.

</details>

**Problem 5.** A 4-by-4 grid of cells over the 2-pixel map holds these counts of 100 dataset points (rows: pixel 1 ranges 0-0.25, 0.25-0.5, 0.5-0.75, 0.75-1 from top; columns: pixel 2, same ranges left to right):

```
 1   0   2  46
 0   1   0   3
 2   0   1   0
41   2   1   0
```

(a) Which cell is most crowded and what share does it hold? (b) What share does the cell containing the point (0.6, 0.6) hold? (c) Do all 16 shares total 1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
counts_table = np.array([
    [1, 0, 2, 46],
    [0, 1, 0, 3],
    [2, 0, 1, 0],
    [41, 2, 1, 0],
])
total_pts = counts_table.sum()
shares_table = counts_table / total_pts

print(f"total points: {total_pts}")
print(f"largest share: {shares_table.max():.2f}")
print(f"share of the cell holding (0.6, 0.6): {shares_table[2, 2]:.2f}")
print(f"sum of all shares: {shares_table.sum():.2f}")
```

The totals per row are 49, 4, 3, 44, so the grand total is 100. (a) The largest count is 46 — pixel 1 in 0-0.25, pixel 2 in 0.75-1 — share $46/100 = 0.46$. (b) The point (0.6, 0.6) falls in the (0.5-0.75, 0.5-0.75) cell, count 1, share $0.01$. (c) The counts sum to 100, so the shares sum to exactly 1.

**Answer:** (a) share 0.46 (dark-left bright-right corner); (b) 0.01 — nearly empty; (c) yes, exactly 1. The table is a hand-built miniature of $p(x)$: two heavy corners, a weightless middle.

</details>

**Problem 6.** Close the lesson and explain the diffusion idea to a friend using the ink-drop story. Your explanation must cover four things: why destruction is easy, why one step is learnable, what exactly gets learned, and how generation works. Write it out (a markdown cell or on paper), then compare with the model answer.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

Model answer, move by move:

1. **Destruction is easy.** An ink drop in water spreads a little each moment; after long enough the glass is uniform pale gray, and every starting drop ends in that same featureless state. Adding a tiny dose of noise to every pixel, repeatedly, does the same to an image — a fixed recipe, no learning needed.
2. **One step is learnable.** Any two consecutive frames of the film look almost identical, so "given the picture at step $t$, estimate step $t-1$" is a small, local correction — not the absurd one-leap request "turn gray water into an ink drop". Training pairs are free: run the destruction recipe on real images.
3. **What gets learned.** One network, one job: given a noisy image and the step number, estimate one step less noisy. The giant leap is never attempted.
4. **How generation works.** Draw pure static (easy — it is mere noise), then apply the learned undo step $T$ times; a thousand tiny reversals add up to one full reversal, and because the starting static is random each time, each output is a genuinely new point from the crowded region of the cloud.

Honesty footnote: "ends in pure noise" is proven in part 6, the exact undo step is derived in part 7, and parts 8-9 reduce learning it to "predict the noise, squared error".

**Answer:** any retelling with those four moves — and the addresses of the proofs — matches.

</details>

## Wrap-up

Verified in this notebook: an image really is a grid of numbers in the 0-1 range (asserted); flattening to a 64-entry vector and back loses nothing (asserted); the dataset is a cloud whose crowded and empty regions are measurable, with cell shares obeying both distribution laws (asserted); and rising doses of noise destroy structure gradually and monotonically (asserted). Part 2 builds the precise language for the cloud's rule — probability — starting from a single die roll.